# TP 1a: LOCALIZED WILSON-COWAN MODEL / RESTING STATE 
## Contact: emre.baspinar@inria.fr

In [ ]:
#################################################################################################################
## Initialization ###############################################################################################
#################################################################################################################

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

#---------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------------------------------#

## Step 1
We define below the sigmoid function, which we use as population transfer function. Vary its parameters and plot the function with varied parameters. Comment on what you observe in the transfer function. How can you interpret these changes in a biological context?

In [ ]:
#################################################################################################################
## Population transfer function: sigmoid ########################################################################
#################################################################################################################

def sigmoid(xi, beta=1.0, xi0=0.0):
    return 1 / (1 + np.exp(-beta * (xi - xi0)))

#---------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------------------------------#


#################################################################################################################
## Plot the transfer function with varied parameters ############################################################
#################################################################################################################

# Define x0 range of xi and set x0 value
xi = np.linspace(0, 20, 400)
x0 = 10

# 1️⃣ Effect of changing beta (slope)
betas = [1]
plt.figure(figsize=(8, 5))
for b in betas:
    plt.plot(xi, sigmoid(xi, beta=b, xi0=x0), label=f"β = {b}")
plt.title("Effect of β on Sigmoid Function")
plt.xlabel("xi")
plt.ylabel("sigmoid(xi)")
plt.legend()
plt.grid(True)
plt.show()

xiForxi0s = np.linspace(0, 20, 400)
# 2️⃣ Effect of changing xi0 (horizontal shift)
xi0s = [8]
plt.figure(figsize=(8, 5))
for x0 in xi0s:
    plt.plot(xiForxi0s, sigmoid(xiForxi0s, beta=1, xi0=x0), label=f"ξ₀ = {x0}")
plt.title("Effect of ξ₀ on Sigmoid Function")
plt.xlabel("xi")
plt.ylabel("sigmoid(xi)")
plt.legend()
plt.grid(True)
plt.show()

#---------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------------------------------#

## Step 2
We define below the localized Wilson-Cowan equations, which we will integrate in time by using odeint command afterwards. Familiarize with the equations, compare them to the localized Wilson-Cowan equations given in the course slides. Why certain parameters are defined separately for excitatory and inhibitory populations, especially the parameters regarding the population transfer functions? What do the connectivity weights represent? Why do the excitatory and inhibitory inputs to the transfer functions have different signs? 

Be careful with the notation used for the connectivity weights!

In [ ]:
#################################################################################################################
## Model equations ##############################################################################################
#################################################################################################################

## Localized Wilson-Cowan equations
def wilson_cowan(u, t, params):
    E, I = u                                                         # Construct the vector for time integration via odeint                                                 
    wee, wei, wie, wii, P, Q, beta_e, beta_i, xi0_e, xi0_i, mu = params        
    
    # Transfer functions
    Se = sigmoid(wee * E - wei * I + P, beta=beta_e, xi0=xi0_e)       # excitatory population transfer function
    Si = sigmoid(wie * E - wii * I + Q, beta=beta_i, xi0=xi0_i)       # inhibitory population transfer function
    
    # Compute the increments of E and I in time  
    dEdt = 1/mu*(-E + (1-E)*Se)                                                  
    dIdt = 1/mu*(-I + (1-I)*Si)
    
    return [dEdt, dIdt]

#---------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------------------------------#

## Step 3
We find below all the parameters of the localized Wilson-Cowan equations described by the command "wilson_cowan". Moreover, we find the initial conditions. Could you comment on the choice of parameters? What does the choice of the external stimulation mean? What is the interpretation of the assigned initial conditions in a biological context? Why are the initial conditions chosen so? What type of neural state do you expect that the localized Wilson-Cowan will generate with these parameters and initial conditions? Comment on these. 

In [ ]:

#################################################################################################################
##  Model parameters and initial conditions: resting state  #####################################################
#################################################################################################################

# Model parameters
params = {
    'wee': 10.0,                             # E <- E connection weight from excitatory neurons to excitatory neurons (recurrent connection)
    'wei': 10.0,                             # E <- I connection weight from inhibitory neurons to excitatory neurons
    'wie': 10.0,                             # I <- E connection weight from excitatory neurons to inhibitory neurons
    'wii': 2.0,                              # I <- I connection weight from inhibitory neurons to inhibitory neurons (recurrent connection)
    'P': 0.0,                                # External stimulus to the excitatory population
    'Q': 0.0,                                # External stimulus to the inihibitory population
    'beta_e': 0.5, 'beta_i': 0.5,            # Nonlinearity sharpness of the transfer functions
    'xi0_e': 10.0, 'xi0_i': 10.0,            # Activity thresholds of the transfer functions
    'mu': 1.0                                # Time constant
}


# Assign the parameter values for the time integration
param_values = list(params.values())

# Initial conditions for resting state: E and I at t=0
u0 = [0.8, 0.8]

#---------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------------------------------#

## Step 4
We simulate the localized Wilson-Cowan model below. By simulation, we refer to an integration of the model equations "wilson-cowan" in time. Simulate the model and observe the result. Comment on the result.

What do the final time, number of samples mean? What do they change in the simulations? Vary these two parameters and observe the changes in the simulation results. Comment on these changes.

In [ ]:
#################################################################################################################
## Simulation of the model ######################################################################################
#################################################################################################################

# Time discretization
T = 10                              # final time
nOfSamples = 1000                   # number of time samples

t = np.linspace(0, T, nOfSamples)   # time vector

# Integrate the system in time
sol = odeint(wilson_cowan, u0, t, args=(param_values,))
E, I = sol.T

#---------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------------------------------------------------#


#########################################################################################
## Plot the simulation results ##########################################################
#########################################################################################

# Plot results
min_y = 0; max_y = 1  # For plot range
plt.figure(figsize=(10, 5))
plt.plot(t, E, label='Excitatory (E)', color='b')
plt.plot(t, I, label='Inhibitory (I)', color='r')
plt.ylim(min_y, max_y)
plt.xlabel('Time')
plt.ylabel('Activity')
plt.title('Wilson-Cowan Model Dynamics')
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

## Step 5
Change the initial conditions and simulate the localized Wilson-Cowan by starting from different initial conditions. Comment on the results.

In [ ]:
#################################################################################################################
## Simulation of the model with different initial conditions ##################################################
#################################################################################################################

# Initial conditions E and I at t=0
u0 = [0.8, 0.8]

# Integrate the system in time
sol = odeint(wilson_cowan, u0, t, args=(param_values,))
E, I = sol.T

#########################################################################################
## Plot the simulation results ##########################################################
#########################################################################################

# Plot results
min_y = 0; max_y = 1  # For plot range
plt.figure(figsize=(10, 5))
plt.plot(t, E, label='Excitatory (E)', color='b')
plt.plot(t, I, label='Inhibitory (I)', color='r')
plt.ylim(min_y, max_y)
plt.xlabel('Time')
plt.ylabel('Activity')
plt.title('Wilson-Cowan Model Dynamics')
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

## Step 6
Change the time constant "mu" and simulate the system. Comment on the changes that you observe when you increase and decrease the time constant. What does the time constant determine in the simulations?

In [ ]:
# Model parameters
params = {
    'wee': 10.0,                             # E <- E connection weight from excitatory neurons to excitatory neurons (strong recurrent excitation)
    'wei': 10.0,                             # E <- I (inhibition onto excitatory)
    'wie': 10.0,                             # I <- E (excitation onto inhibitory)
    'wii': 2.0,                              # I <- I (mild self-inhibition)
    'P': 0.0,                                # External stimulus to the excitatory population
    'Q': 0.0,                                # External stimulus to the inihibitory population
    'beta_e': 0.5, 'beta_i': 0.5,            # Nonlinearity sharpness of the transfer functions
    'xi0_e': 10.0, 'xi0_i': 10.0,            # Activity thresholds of the transfer functions
    'mu': 1.0                                # Time constant
}


# Assign the parameter values for the time integration
param_values = list(params.values())

# Initial conditions for resting state: E and I at t=0
u0 = [0.8, 0.8]

T = 20                              # final time
nOfSamples = 1000                   # number of time samples

t = np.linspace(0, T, nOfSamples)   # time vector

# Integrate the system in time
sol = odeint(wilson_cowan, u0, t, args=(param_values,))
E, I = sol.T

#########################################################################################
## Plot the simulation results ##########################################################
#########################################################################################

# Plot results
min_y = 0; max_y = 1  # For plot range
plt.figure(figsize=(10, 5))
plt.plot(t, E, label='Excitatory (E)', color='b')
plt.plot(t, I, label='Inhibitory (I)', color='r')
plt.ylim(min_y, max_y)
plt.xlabel('Time')
plt.ylabel('Activity')
plt.title('Wilson-Cowan Model Dynamics')
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()